# Interactive Goal-Oriented RL Chatbot

This notebook lets you interact with the trained DQN-based goal-oriented chatbot for movie ticket booking.

You can:
- **Watch** the user simulator converse with the agent
- **Chat** with the agent yourself via an interactive widget
- **Inspect** the agent's state, Q-values and decision-making in real time

## 1. Setup & Imports

In [ ]:
import os, sys, json, pickle, copy, random, importlib
import numpy as np

# Ensure the repo root is on the path
REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))  # notebook dir
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Import repo modules, then force-reload them so any on-disk edits are picked up
import user_simulator, error_model_controller, dqn_agent, state_tracker, utils, dialogue_config
for _mod in [dialogue_config, utils, dqn_agent, state_tracker, error_model_controller, user_simulator]:
    importlib.reload(_mod)

from user_simulator import UserSimulator
from error_model_controller import ErrorModelController
from dqn_agent import DQNAgent
from state_tracker import StateTracker
from utils import remove_empty_slots
from dialogue_config import all_intents, all_slots, usersim_default_key, FAIL, SUCCESS, NO_OUTCOME

print('All imports successful.')

All imports successful.


## 2. Load Configuration & Data

Edit `WEIGHTS_PATH` below to point to your trained model weights (without the `_beh.h5` / `_tar.h5` suffix).

In [26]:
# ── One-time fix: strip CRLF from pickle files (Python 2 → Python 3) ──
# Only needs to be run once. Safe to re-run — it's idempotent.
import os

def fix_pickle_crlf(path):
    with open(path, 'rb') as f:
        content = f.read()
    fixed = b'\n'.join(content.splitlines())
    with open(path, 'wb') as f:
        f.write(fixed)
    print(f'Fixed: {path}')

for pkl in ['data/movie_db.pkl', 'data/movie_dict.pkl', 'data/movie_user_goals.pkl']:
    fix_pickle_crlf(pkl)

print('Pickle files ready.')

Fixed: data/movie_db.pkl
Fixed: data/movie_dict.pkl
Fixed: data/movie_user_goals.pkl
Pickle files ready.


In [27]:
# ---------- CONFIGURATION ----------
CONSTANTS_FILE = os.path.join(REPO_ROOT, 'constants.json')
WEIGHTS_PATH   = 'weights/model.h5'   # Agent will load weights/model_beh.h5 & weights/model_tar.h5
# -----------------------------------

with open(CONSTANTS_FILE) as f:
    constants = json.load(f)

# Override weights path so the agent loads the pretrained model
constants['agent']['load_weights_file_path'] = WEIGHTS_PATH
# Set epsilon to 0 so the agent always exploits (no random actions)
constants['agent']['epsilon_init'] = 0.0

# Load pickle data  (encoding='latin1' is required to unpickle Python 2 pickles in Python 3)
database   = pickle.load(open(constants['db_file_paths']['database'],   'rb'), encoding='latin1')
db_dict    = pickle.load(open(constants['db_file_paths']['dict'],       'rb'), encoding='latin1')
user_goals = pickle.load(open(constants['db_file_paths']['user_goals'], 'rb'), encoding='latin1')

remove_empty_slots(database)

print(f'Database entries : {len(database)}')
print(f'User goals       : {len(user_goals)}')
print(f'Dict keys        : {list(db_dict.keys())}')

Database entries : 991
User goals       : 128
Dict keys        : ['city', 'numberofpeople', 'theater', 'description', 'zip', 'numberofkids', 'distanceconstraints', 'critic_rating', 'price', 'greeting', 'actor', 'date', 'state', 'other', 'mpaa_rating', 'starttime', 'theater_chain', 'genre', 'video_format', 'moviename']


## 3. Initialize Components

In [ ]:
user_sim      = UserSimulator(user_goals, constants, database)
emc           = ErrorModelController(db_dict, constants)
state_tracker = StateTracker(database, constants)
dqn_agent     = DQNAgent(state_tracker.get_state_size(), constants)

print(f'State size       : {state_tracker.get_state_size()}')
print(f'Number of actions: {dqn_agent.num_actions}')
print('Components initialised. Weights loaded.' if WEIGHTS_PATH else 'Components initialised (no weights).')

ValueError: Argument(s) not recognized: {'lr': 0.001}

---
## 4. Watch the Agent vs User Simulator

Run automated episodes and see the full conversation, the user's hidden goal, and whether the agent succeeded.

In [ ]:
def run_episode(verbose=True):
    """
    Run a single episode with the user simulator and return a transcript.
    
    Returns:
        transcript (list[dict]): list of actions per turn
        success (bool)
        total_reward (float)
        goal (dict): the user simulator's hidden goal
    """
    # Reset
    state_tracker.reset()
    user_action = user_sim.reset()
    goal = copy.deepcopy(user_sim.goal)
    emc.infuse_error(user_action)
    state_tracker.update_state_user(user_action)
    dqn_agent.reset()

    transcript = [{'turn': 0, 'speaker': 'User', **copy.deepcopy(user_action)}]
    done = False
    total_reward = 0
    success = False

    while not done:
        # Agent acts
        state = state_tracker.get_state()
        action_index, agent_action = dqn_agent.get_action(state)
        state_tracker.update_state_agent(agent_action)
        transcript.append({'turn': state_tracker.round_num, 'speaker': 'Agent', **copy.deepcopy(agent_action)})

        # User responds
        user_action, reward, done, succ = user_sim.step(agent_action)
        total_reward += reward
        if not done:
            emc.infuse_error(user_action)
        state_tracker.update_state_user(user_action)
        transcript.append({'turn': state_tracker.round_num, 'speaker': 'User', **copy.deepcopy(user_action)})
        success = succ

    return transcript, success, total_reward, goal


def pretty_print_episode(transcript, success, total_reward, goal):
    """Nicely format and print a conversation."""
    print('=' * 70)
    print('USER GOAL')
    print(f"  Inform slots : {goal['inform_slots']}")
    print(f"  Request slots: {goal['request_slots']}")
    print('-' * 70)
    for t in transcript:
        speaker = t['speaker']
        intent  = t.get('intent', '')
        informs = t.get('inform_slots', {})
        reqs    = t.get('request_slots', {})
        parts = [f'[{speaker:>5}] intent={intent}']
        if informs:
            parts.append(f'  inform={informs}')
        if reqs:
            parts.append(f'  request={reqs}')
        print(' | '.join(parts))
    print('-' * 70)
    result_str = 'SUCCESS' if success else 'FAILURE'
    print(f'Result: {result_str}  |  Total reward: {total_reward}')
    print('=' * 70)

print('Helper functions defined.')

In [ ]:
# Run and display a single episode
transcript, success, total_reward, goal = run_episode()
pretty_print_episode(transcript, success, total_reward, goal)

In [ ]:
# Run N episodes and compute aggregate stats
NUM_EPISODES = 200

successes = 0
rewards   = []
lengths   = []

for _ in range(NUM_EPISODES):
    t, s, r, g = run_episode(verbose=False)
    successes += int(s)
    rewards.append(r)
    lengths.append(len(t))

print(f'Episodes      : {NUM_EPISODES}')
print(f'Success rate  : {successes / NUM_EPISODES:.2%}')
print(f'Avg reward    : {np.mean(rewards):.2f} ± {np.std(rewards):.2f}')
print(f'Avg turns     : {np.mean(lengths):.1f}')

---
## 5. Interactive Chat — Talk to the Agent Yourself

Use the cell below to have a conversation with the agent **turn by turn**.  
You type your action in the format: `intent/inform_key: value, .../request_key, ...`

### Action format examples
| Input | Meaning |
|---|---|
| `request/moviename: room, date: friday/starttime, city` | Request with informs and requests |
| `inform/moviename: zootopia/` | Inform a movie name |
| `request//starttime` | Request with no informs |
| `done//` | End the conversation |

In [ ]:
from dialogue_config import usersim_intents, all_slots as ALL_SLOTS

def parse_user_input(input_string):
    """
    Parse a user input string into a dialogue action dict.
    
    Format: intent/inform_key: value, .../request_key, ...
    Examples:
        request/moviename: room, date: friday/starttime, city
        inform/moviename: zootopia/
        done//
    """
    response = {'intent': '', 'inform_slots': {}, 'request_slots': {}}
    chunks = input_string.strip().split('/')
    if len(chunks) != 3:
        return None, 'Input must have exactly 3 parts separated by /'
    
    # Intent
    intent = chunks[0].strip()
    if intent not in usersim_intents:
        return None, f'Invalid intent "{intent}". Must be one of: {usersim_intents}'
    response['intent'] = intent
    
    # Inform slots
    if chunks[1].strip():
        for pair in chunks[1].split(','):
            pair = pair.strip()
            if ':' not in pair:
                return None, f'Inform slot "{pair}" missing ":" separator'
            key, value = pair.split(':', 1)
            key, value = key.strip(), value.strip()
            if key not in ALL_SLOTS:
                return None, f'Unknown slot "{key}". Valid slots: {ALL_SLOTS}'
            response['inform_slots'][key] = value
    
    # Request slots
    if chunks[2].strip():
        for req in chunks[2].split(','):
            req = req.strip()
            if req not in ALL_SLOTS:
                return None, f'Unknown request slot "{req}". Valid slots: {ALL_SLOTS}'
            response['request_slots'][req] = 'UNK'
    
    return response, None

print('Parser ready. Valid intents:', usersim_intents)
print('Valid slots:', ALL_SLOTS)

In [ ]:
def interactive_chat():
    """
    Run a full interactive conversation with the agent.
    You provide user actions, and the agent responds using its trained policy.
    After each agent response, enter 'success' (1), 'fail' (-1), or 'continue' (0).
    """
    # Reset everything
    state_tracker.reset()
    dqn_agent.reset()
    
    print('=' * 60)
    print('INTERACTIVE CHAT — Type your actions to converse with the agent')
    print('Type "quit" to exit at any time.')
    print('=' * 60)
    
    # --- Initial user action ---
    print('\n>> Enter your FIRST action (e.g. request/moviename: zootopia/starttime, city):')
    first_input = input('You: ')
    if first_input.strip().lower() == 'quit':
        print('Exited.')
        return
    user_action, err = parse_user_input(first_input)
    if err:
        print(f'Error: {err}')
        return
    
    state_tracker.update_state_user(user_action)
    print(f'  [You]   intent={user_action["intent"]}  inform={user_action["inform_slots"]}  request={user_action["request_slots"]}')
    
    done = False
    turn = 0
    while not done:
        turn += 1
        # Agent acts
        state = state_tracker.get_state()
        action_index, agent_action = dqn_agent.get_action(state)
        state_tracker.update_state_agent(agent_action)
        print(f'\n  [Agent] intent={agent_action["intent"]}  inform={agent_action["inform_slots"]}  request={agent_action["request_slots"]}')
        
        # Show Q-values for the current state
        q_values = dqn_agent._dqn_predict_one(state)
        top5_idx = np.argsort(q_values)[-5:][::-1]
        print('  Top-5 Q-values:')
        for idx in top5_idx:
            act = dqn_agent.possible_actions[idx]
            print(f'    [{idx:2d}] Q={q_values[idx]:+.3f}  {act}')
        
        # Check if agent said done
        if agent_action['intent'] == 'done':
            print('\nAgent ended the conversation.')
            break
        
        if turn >= constants['run']['max_round_num']:
            print('\nMax rounds reached. Conversation over.')
            break
        
        # User responds
        print(f'\n>> Enter your action (turn {turn + 1}):')
        user_input = input('You: ')
        if user_input.strip().lower() == 'quit':
            print('Exited.')
            return
        user_action, err = parse_user_input(user_input)
        if err:
            print(f'Error: {err}')
            continue
        
        state_tracker.update_state_user(user_action)
        print(f'  [You]   intent={user_action["intent"]}  inform={user_action["inform_slots"]}  request={user_action["request_slots"]}')
        
        if user_action['intent'] == 'done':
            print('\nYou ended the conversation.')
            done = True
    
    print('\nConversation finished.')
    print('Current inform slots tracked:', state_tracker.current_informs)

In [ ]:
# Start an interactive conversation — run this cell and type in the output area
interactive_chat()

---
## 6. Inspect Agent Internals

Peek at the Q-values for a specific user action to understand _why_ the agent picks a particular response.

In [ ]:
def inspect_q_values(user_action_str):
    """
    Given a user action string, show the agent's Q-value ranking for all possible responses.
    """
    state_tracker.reset()
    dqn_agent.reset()
    
    user_action, err = parse_user_input(user_action_str)
    if err:
        print(f'Error: {err}')
        return
    
    state_tracker.update_state_user(user_action)
    state = state_tracker.get_state()
    q_values = dqn_agent._dqn_predict_one(state)
    
    sorted_idx = np.argsort(q_values)[::-1]
    
    print(f'User action: {user_action}')
    print(f'\n{"Rank":>4}  {"Index":>5}  {"Q-value":>10}  Action')
    print('-' * 80)
    for rank, idx in enumerate(sorted_idx, 1):
        act = dqn_agent.possible_actions[idx]
        marker = ' <-- CHOSEN' if rank == 1 else ''
        print(f'{rank:4d}  {idx:5d}  {q_values[idx]:+10.4f}  {act}{marker}')

# Example: see what the agent would do if the user requests a movie
inspect_q_values('request/moviename: zootopia/starttime')

---
## 7. Explore the Movie Database

In [ ]:
# Show a few sample database entries
sample_keys = list(database.keys())[:5]
for k in sample_keys:
    print(f'Entry {k}: {database[k]}')
    print()

In [ ]:
# Show a few sample user goals
for i, goal in enumerate(user_goals[:3]):
    print(f'Goal {i}: {goal}')
    print()